# 01 — Exploratory Data Analysis

**Goals of this notebook**
1. Understand the universe: coverage, liquidity, return distributions
2. Document data quality issues and survivorship bias
3. Examine cross-sectional return dispersion (prerequisite for ranking)
4. Check macro series for structural breaks

**Run fetch_data.py first:** `python data/fetch_data.py`

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config as cfg
from src.plotting import plot_return_distribution, plot_universe_coverage

PROCESSED = cfg.processed_dir()
RAW = cfg.PROJECT_ROOT / cfg.get('data.raw_dir')
print('Data dirs OK')

## 1. Load Raw Price Data

In [ ]:
prices = pd.read_parquet(RAW / 'prices.parquet')
print(f'Shape: {prices.shape}')
print(f'Tickers: {prices.index.get_level_values("Ticker").nunique()}')
print(f'Date range: {prices.index.get_level_values("Date").min()} → {prices.index.get_level_values("Date").max()}')
prices.head()

## 2. Universe Coverage

In [ ]:
fig = plot_universe_coverage(prices)
plt.show()

# Tickers with sparse data — flag as potentially problematic
close = prices['close'].unstack('Ticker')
coverage = close.notna().sum()
sparse = coverage[coverage < coverage.mean() - coverage.std()]
print(f'Potentially sparse tickers ({len(sparse)}):', sparse.sort_values().head(10).to_dict())

## 3. Return Distribution — Fat Tails & Skewness

In [ ]:
# Daily log returns for the full universe
log_ret = np.log(close / close.shift(1)).stack()
log_ret = log_ret.dropna()
log_ret.name = 'log_return'

print(f'Universe daily returns — descriptive stats:')
print(log_ret.describe().to_string())
print(f'Skewness : {log_ret.skew():.3f}')
print(f'Kurtosis : {log_ret.kurtosis():.3f} (excess — normal=0)')

fig = plot_return_distribution(log_ret, ticker_or_label='Full Universe Daily Returns')
plt.show()

## 4. Cross-Sectional Return Dispersion

In [ ]:
# 21-day forward returns — the ranking target
fwd_21 = (close.shift(-22) / close.shift(-1) - 1)

# Cross-sectional std per date = how much spread exists to exploit
cs_std = fwd_21.std(axis=1).dropna()

fig, ax = plt.subplots(figsize=(12, 4))
cs_std.rolling(21).mean().plot(ax=ax, color='#185FA5', linewidth=1.5,
                                label='21-day MA of CS std')
ax.axhline(cs_std.mean(), color='#A32D2D', linestyle='--', linewidth=1,
            label=f'Mean = {cs_std.mean():.3f}')
ax.set_title('Cross-Sectional Dispersion of 21-Day Forward Returns\n'
             '(Higher = more opportunity for ranking to matter)')
ax.set_ylabel('Std of forward returns')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nKey insight: CS dispersion = {cs_std.mean():.3f} on average')
print('This represents the theoretical max spread a perfect ranker could capture')

## 5. Macro Data Check

In [ ]:
try:
    macro = pd.read_parquet(RAW / 'macro.parquet')
    print('Macro shape:', macro.shape)
    print(macro.describe())

    fig, axes = plt.subplots(len(macro.columns), 1, figsize=(12, 2.5 * len(macro.columns)),
                              sharex=True)
    if len(macro.columns) == 1:
        axes = [axes]
    for ax, col in zip(axes, macro.columns):
        macro[col].plot(ax=ax, color='#185FA5', linewidth=1)
        ax.set_title(col)
        ax.set_ylabel('')
    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print('Macro data not found — run fetch_data.py with FRED_API_KEY set')

## Summary

Fill in after running:
- Universe: N tickers, date range YYYY-MM-DD to YYYY-MM-DD
- Data quality issues: (any sparse tickers?)
- Return distribution: fat tails confirmed (kurtosis >> 0)
- CS dispersion: average Xpct — sufficient spread for ranking

→ Proceed to `02_feature_eng.ipynb`